## Resumen
Este notebook intenta replicar la logica de GlobalDist PPP 2021 usando solo campos disponibles, separando claramente que componentes son replicables y cuales requieren archivos intermedios no presentes.

# Replicating Global 1000 Bins Logic (PPP 2021) with Available Fields

This notebook replicates as much as possible of the Stata pipeline using files available in this workspace.

Main limitation: `CollapsedDistributions.dta` is not available, so the exact pre-GlobalDist filling step cannot be reproduced 1:1.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

In [ ]:
ROOT = Path('.').resolve()
INPUT = ROOT / '01-input'

fillgaps_candidates = [
    INPUT / '20260922' / 'lineup' / 'fillgaps.dta',
    INPUT / 'nuevosyuki' / 'fillgaps.dta',
]

globaldist_candidates = [
    INPUT / '20260922' / 'lineup' / 'GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.dta',
    INPUT / 'nuevosyuki' / 'GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.dta',
]

fillgaps_path = next((p for p in fillgaps_candidates if p.exists()), None)
globaldist_path = next((p for p in globaldist_candidates if p.exists()), None)
interpolated_path = INPUT / 'interpolated_means.dta'
collapsed_path = next((p for p in INPUT.rglob('*CollapsedDistributions*.dta')), None)

fillgaps_path, globaldist_path, interpolated_path, collapsed_path

In [ ]:
if fillgaps_path is None or globaldist_path is None:
    raise FileNotFoundError('Missing required PPP 2021 inputs: fillgaps or GlobalDist.')

fillgaps = pd.read_stata(fillgaps_path, convert_categoricals=False)
globaldist = pd.read_stata(globaldist_path, convert_categoricals=False)
interpolated = pd.read_stata(interpolated_path, convert_categoricals=False)

for df in (fillgaps, globaldist, interpolated):
    if 'year' in df.columns:
        df['year'] = df['year'].astype(int)

print('fillgaps shape:', fillgaps.shape)
print('globaldist shape:', globaldist.shape)
print('interpolated shape:', interpolated.shape)

## 1) PPP 2021 consistency checks

Stata script uses an explicit `ppp_year(...)` option. In available files, PPP is identified indirectly via vintage and interpolation metadata.

In [ ]:
ppp_cols_interp = [c for c in interpolated.columns if 'ppp' in c.lower() or 'cpi' in c.lower()]
ppp_cols_fg = [c for c in fillgaps.columns if 'ppp' in c.lower() or 'cpi' in c.lower()]
ppp_cols_gd = [c for c in globaldist.columns if 'ppp' in c.lower() or 'cpi' in c.lower()]

print('GlobalDist pipvintage values:')
print(globaldist['pipvintage'].dropna().value_counts().head(10))
print('\nInterpolated PPP/CPI columns:', ppp_cols_interp)
print('Fillgaps PPP/CPI columns:', ppp_cols_fg)
print('GlobalDist PPP/CPI columns:', ppp_cols_gd)

## 2) Build `pop_all` analog from fillgaps

Equivalent intent to Stata section that prepares population by country/year/reporting_level and region.

In [ ]:
pop_all = fillgaps[[
    'country_code', 'region_code', 'reporting_level', 'year', 'population'
]].dropna(subset=['country_code', 'year', 'population']).copy()

# Keep national plus ARG/SUR urban/rural behavior from Stata logic.
pop_all = pop_all[(pop_all['reporting_level'] == 'national') | (pop_all['country_code'].isin(['ARG', 'SUR']))]
pop_all = pop_all[(pop_all['year'] > 1980) & (pop_all['year'] < 2020)]

# Stata converts to millions.
pop_all['pop_millions'] = pop_all['population'] / 1_000_000

# One row per key.
pop_all = pop_all.drop_duplicates(['country_code', 'reporting_level', 'year'])

pop_all.head(10)

## 3) Build working 1000-bin structure from GlobalDist

Stata uses `obs` as bin number. Here we map `quantile` -> `obs` and assume national coverage in final GlobalDist.

In [ ]:
gd = globaldist.copy()
gd = gd.rename(columns={'quantile': 'obs'})
gd['obs'] = gd['obs'].astype(int)
gd['reporting_level'] = 'national'

# Bring region_code as in Stata output shape.
gd = gd.merge(
    pop_all[['country_code', 'year', 'region_code']].drop_duplicates(['country_code', 'year']),
    left_on=['code', 'year'],
    right_on=['country_code', 'year'],
    how='left'
).drop(columns=['country_code'])

gd[['year', 'code', 'region_code', 'obs', 'welf', 'pop']].head(10)

## 4) Quality checks vs Stata intent

Check each country-year has 1000 bins and no missing welfare in final GlobalDist.

In [ ]:
bins_per_cy = gd.groupby(['code', 'year'], as_index=False).agg(
    n_bins=('obs', 'nunique'),
    miss_welf=('welf', lambda s: s.isna().sum())
)

print('Country-years with n_bins != 1000:', (bins_per_cy['n_bins'] != 1000).sum())
print('Country-years with missing welf > 0:', (bins_per_cy['miss_welf'] > 0).sum())

bins_per_cy.sort_values(['n_bins', 'miss_welf', 'code', 'year']).head(20)

## 5) What can and cannot be replicated exactly

### Replicated with available fields
- PPP-2021 vintage consistency checks (`pipvintage`, interpolation PPP fields).
- Population lookup analog (`pop_all`) from `fillgaps`.
- Final 1000-bin structure checks (`obs`, `welf`, `pop`) from PPP-2021 GlobalDist.

### Not replicable exactly with current files
- `CollapsedDistributions.dta` step (pre-final distribution with missing welfare to fill).
- Exact ARG/SUR urban-rural-to-national collapse step from raw intermediate distributions.

If `CollapsedDistributions.dta` (PPP-2021 vintage) is provided, this notebook can be extended to a near 1:1 Stata replication.